# AI-Powered Ticket Processing Pipeline - Phase1: Prompt Testing
**Objective:** Evaluate LLM prompt strategies using Chain of Thought (CoT) and enforce structured JSON outputs using Pydantic on the 50-sample dataset.

## 1. Environment Setup and Configuration
Importing required dependencies and configuring the LLM client and projects paths.

In [2]:
import os
import json
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI
from tqdm import tqdm

In [3]:
try:
  from google.colab import userdata
  API_KEY = userdata.get("OPENAI_API_KEY")

except:
  API_KEY = os.getenv("OPENAI_API_KEY")

CONFIG = {
    "input_sample_file" : "data/poc_sample_50.csv",
    "output_results_file" : "data/poc_results_50.csv",
    "text_col" : "response",
    "model_name" : "gpt-5.6-sol",
    "base_url" : "https://api.gapgpt.app/v1",
    "api_key" : API_KEY
}

In [4]:
#  Initialize LLM client
client = OpenAI(
    api_key = CONFIG["api_key"],
    base_url = CONFIG["base_url"]
)

## 2. Load Sampled Dataset
Loading the strategic 50-sample dataset generated in the previous PoC phase.

In [5]:
df_sample = pd.read_csv(CONFIG["input_sample_file"])

print(f"Loaded dataset shape:{df_sample.shape}")
df_sample.head()

Loaded dataset shape:(50, 7)


,flags,instruction,category,intent,response,text_length,noise_score
0,BCL,"I want to check your money back guarantee, I n...",REFUND,check_refund_policy,Definitely! I completely understand your desir...,2424,0.035066
1,BL,help me seeing what hours I can contact custom...,CONTACT,contact_customer_service,Grateful for your contact! I get the sense tha...,457,0.024070
2,BCIL,"I have got to use the standard profile, can I ...",ACCOUNT,switch_account,For sure! I'm here to provide you with the sup...,831,0.108303
3,BL,I want assistance making a complaint against y...,FEEDBACK,complaint,I'm sorry to hear that you've had a negative e...,630,0.019048
4,BILQ,can ya help me to check ur reimbursement policy,REFUND,check_refund_policy,For sure! I understand your request to check o...,2366,0.032544


## 3. Define Output Schema (Pydantic)
Defin the expected JSON structure for the LLM output, including Moderation, Classification, Sentiment, and Summarization.

In [6]:
class TicketProcessingResult(BaseModel):
    is_safe: bool = Field(description="True if the ticket contains no harmful, offensive, or unsafe content (Moderation).")
    category: str = Field(description="The main category of the ticket (e.g., Billing, Technical Support, Account, General Inquiry).")
    sentiment: str = Field(description="The emotional tone of the ticket: Positive, Neutral, or Negative.")
    summary: str = Field(description="A concise, one-sentence summary of the user's issue.")
    suggested_action: str = Field(description="A brief recommended next step for the support agent.")
    reasoning: str = Field(description="Chain of Thought: Step-by-step reasoning explaining how the category, sentiment, and safety were determined.")

## 4. Prompt Engineering (System & User Prompts)
Designing the Chain of Though (CoT) system prompt to instruct the model.

In [7]:
SYSTEM_PROMPT = """You are an expert AI customer support assistant.
Your task is to analyze customer support tickets and extract key information.

Analyze the ticket step-by-step (Chain of Though):
1. First, check if the content is safe and appropriate (Moderation).
2. Determine the core topic to assign a category (Classification).
3. Evaluate the tone of the text (Sentiment Analysis).
4. Summarize the issue in one sentence (Summarization).
5. Suggest a logical next step to resolve the issue.

You MUST return your response entirely in valid JSON format matching the exact keys provided by the user. Do not include Markdown blocks like   json.
"""

In [8]:
def create_user_prompt(ticket_text:str) -> str:
  schema = TicketProcessingResult.model_json_schema()
  return f"Please analyze the following ticket and return a JSON object matching this schema:\n\n{json.dumps(schema, indent=2)}\n\nTicket Text:\n\"\"\"{ticket_text}\"\"\""
